# Short-Volshock Teacher Reward / Checkpoint Screen v1

This notebook converts the latest metric-aligned short-volshock teacher handoff into a reward/checkpoint screening artifact. It is a pre-training diagnostic only: the teacher path may guide checkpoint scoring or fold-local target generation, but it is not a promoted actor and not permission for longer PPO.

Governance:

- Baselines: canonical production/reference context `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison teacher source `short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1` and source candidate `multi_obj_floor45_cap20_riskfirst`.
- Role: `diagnose`.
- Target window: final retention requires `wf_2020`, `wf_2022`, and `wf_2023_2024`; official replay gate uses `relative_return >= +30pp` and `MaxDD <= 15%`.
- Changed layer: `reward`, `checkpoint_selection`, `model_selection`; no runtime overlay promotion.
- Hypothesis: a metric-aligned short-volshock teacher can define a drawdown-drift/checkpoint objective, but missing `wf_2020` and any remaining near-miss must block actor retention.
- Failure mode: copying evaluation-window positions into labels or using a two-fold near-miss as promotion evidence.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "soxl_quant").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "soxl_quant").exists():
    raise RuntimeError("could not locate repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.reward_checkpoint_screen import (
    RewardCheckpointScreenConfig,
    render_reward_checkpoint_screen_review,
    score_reward_checkpoint_candidates,
    summarize_reward_checkpoint_screen,
)

CARD = ROOT / "runtime" / "reward" / "card" / "short_volshock_teacher_reward_checkpoint_screen_v1"
TEACHER = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1"
OLD_AUDIT = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_drawdown_drift_feature_audit_v1"
FEATURE_HANDOFF = ROOT / "runtime" / "model_selection" / "card" / "warm_start_official_failure_handoff_v1"
OFFICIAL_FEATURE_AUDIT = ROOT / "runtime" / "model_selection" / "card" / "official_replay_bad_interval_feature_audit_v1"

CARD.mkdir(parents=True, exist_ok=True)
created_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
created_at


'2026-06-16T03:59:21+00:00'

## Load Teacher Inputs

The teacher card is source-of-truth for the selected metric-aligned near-miss. Its recipe explicitly forbids actor promotion, runtime overlay promotion, evaluation-window label copying, or longer-PPO permission.

In [2]:
teacher_summary = json.loads((TEACHER / "summary.json").read_text())
teacher_recipe = json.loads((TEACHER / "teacher_checkpoint_bc_recipe.json").read_text())
fold_metrics = pd.read_csv(TEACHER / "selected_teacher_fold_metrics.csv")
trace_absorption = pd.read_csv(TEACHER / "teacher_trace_absorption.csv")
failing_interval = pd.read_csv(TEACHER / "failing_interval_summary.csv")

assert teacher_summary["decision"] == "short_volshock_metric_aligned_teacher_ready_no_promotion"
assert teacher_recipe["leakage_guard"]["next_target_generation_must_be_fold_local"] is True
assert teacher_recipe["leakage_guard"]["copy_selected_daily_positions_into_training_labels"] is False
assert "permission_for_longer_ppo" in teacher_recipe["do_not_use_as"]

fold_metrics[["fold", "variant", "teacher_status", "official_warm_relative_return", "official_warm_max_dd", "relative_gap_pp", "max_dd_gap_pct"]]


,fold,variant,teacher_status,official_warm_relative_return,official_warm_max_dd,relative_gap_pp,max_dd_gap_pct
0,wf_2022,short_volshock_cap05_hit2,pass,83.564183,8.746809,0.000000,0.000000
1,wf_2023_2024,short_volshock_cap05_hit2,near_miss,29.817263,15.107381,0.182737,0.107381


## Build Gate-Gap Inputs

The score helper requires one row per candidate/window. We compare the source official replay path with the selected teacher path and add explicit missing-window rows for `wf_2020`. Missing coverage is scored as a coverage failure, not as a hidden pass.

In [3]:
target = teacher_recipe["target_contract"]
target_rel = float(target["official_eval_relative_return_pp"])
target_dd = float(target["official_eval_max_dd_pct"])
required_windows = list(target["final_required_windows"])
missing_windows = set(target.get("missing_final_required_windows") or [])

selected_candidate = "teacher_short_volshock_cap05_hit2"
source_candidate = "source_multi_obj_floor45_cap20_riskfirst"

def gate_row(profile: str, window: str, rel: float, max_dd: float, status: str | None = None) -> dict[str, object]:
    rel_gap = max(0.0, target_rel - float(rel))
    dd_gap = max(0.0, float(max_dd) - target_dd)
    gate_pass = rel_gap == 0.0 and dd_gap == 0.0
    if status is None:
        if gate_pass:
            failure_type = "pass"
        elif rel_gap > 0 and dd_gap > 0:
            failure_type = "joint_tradeoff"
        elif rel_gap > 0:
            failure_type = "underparticipation"
        else:
            failure_type = "drawdown_control"
    else:
        failure_type = status
    return {
        "profile": profile,
        "window": window,
        "relative_return": float(rel),
        "max_dd": float(max_dd),
        "relative_gap": float(rel_gap),
        "max_dd_gap": float(dd_gap),
        "gate_pass": bool(gate_pass),
        "failure_type": failure_type,
    }

rows: list[dict[str, object]] = []
for row in fold_metrics.to_dict("records"):
    rows.append(
        gate_row(
            selected_candidate,
            str(row["fold"]),
            float(row["official_warm_relative_return"]),
            float(row["official_warm_max_dd"]),
            "pass" if str(row["teacher_status"]) == "pass" else None,
        )
    )
    rows.append(
        gate_row(
            source_candidate,
            str(row["fold"]),
            float(row["source_official_relative_return"]),
            float(row["source_official_max_dd"]),
        )
    )

for window in missing_windows:
    for candidate in (selected_candidate, source_candidate):
        rows.append(
            {
                "profile": candidate,
                "window": window,
                "relative_return": 0.0,
                "max_dd": target_dd,
                "relative_gap": target_rel,
                "max_dd_gap": 0.0,
                "gate_pass": False,
                "failure_type": "coverage_missing",
            }
        )

gate_gaps = pd.DataFrame(rows).sort_values(["profile", "window"]).reset_index(drop=True)
gate_gaps.to_csv(CARD / "gate_gaps.csv", index=False)
gate_gaps


,profile,window,relative_return,max_dd,relative_gap,max_dd_gap,gate_pass,failure_type
0,source_multi_obj_floor45_cap20_riskfirst,wf_2020,0.000000,15.000000,30.000000,0.000000,False,coverage_missing
1,source_multi_obj_floor45_cap20_riskfirst,wf_2022,59.388029,30.626799,0.000000,15.626799,False,drawdown_control
2,source_multi_obj_floor45_cap20_riskfirst,wf_2023_2024,30.262521,16.122029,0.000000,1.122029,False,drawdown_control
3,teacher_short_volshock_cap05_hit2,wf_2020,0.000000,15.000000,30.000000,0.000000,False,coverage_missing
4,teacher_short_volshock_cap05_hit2,wf_2022,83.564183,8.746809,0.000000,0.000000,True,pass
5,teacher_short_volshock_cap05_hit2,wf_2023_2024,29.817263,15.107381,0.182737,0.107381,False,joint_tradeoff


## Short-Volshock Consistency And Feature Context

The trace absorption data is not a BC slice-fit score. It is used here only as a diagnostic consistency proxy that the selected teacher actually identifies short-volshock deleveraging days. Poor-window clusters and feature groups come from the prior short-volshock drawdown-drift audit, so they can only shape objective terms, not promotion.

In [4]:
trace_summary = trace_absorption.copy()
selected_trace = trace_summary[trace_summary["variant"].astype(str).eq("short_volshock_cap05_hit2")]
short_samples = int(selected_trace["overlay_gate_ready_days"].sum())
consistency = float(selected_trace["cap_changed_share_of_ready"].mean())
slice_scores = pd.DataFrame(
    [
        {
            "candidate": selected_candidate,
            "mean_short_volshock_accuracy": consistency,
            "short_volshock_target_samples": short_samples,
            "slice_gate_pass_count": int((fold_metrics["teacher_status"].astype(str) == "pass").sum()),
            "score_interpretation": "trace_consistency_proxy_not_bc_label_accuracy",
        }
    ]
)
slice_scores.to_csv(CARD / "slice_scores.csv", index=False)

poor_window_clusters = pd.read_csv(OLD_AUDIT / "poor_window_clusters.csv")
feature_frames = []
if (FEATURE_HANDOFF / "feature_group_consensus.csv").exists():
    feature_frames.append(pd.read_csv(FEATURE_HANDOFF / "feature_group_consensus.csv")[["feature_group"]])
if (OFFICIAL_FEATURE_AUDIT / "official_interval_feature_groups.csv").exists():
    official_groups = pd.read_csv(OFFICIAL_FEATURE_AUDIT / "official_interval_feature_groups.csv")[["feature_group"]].copy()
    official_groups["feature_group"] = official_groups["feature_group"].replace(
        {
            "trend_or_momentum": "trend_momentum",
            "volatility_or_stress": "volatility_regime",
        }
    )
    feature_frames.append(official_groups)
feature_groups = pd.concat(feature_frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
feature_groups.to_csv(CARD / "feature_groups.csv", index=False)

slice_scores, poor_window_clusters[["cluster_id", "dominant_failure_type"]].head(), feature_groups.head(10)


(                           candidate  mean_short_volshock_accuracy  \
 0  teacher_short_volshock_cap05_hit2                      0.953125   
 
    short_volshock_target_samples  slice_gate_pass_count  \
 0                            300                      1   
 
                             score_interpretation  
 0  trace_consistency_proxy_not_bc_label_accuracy  ,
   cluster_id dominant_failure_type
 0       C004    underparticipation
 1       C002    underparticipation
 2       C003    underparticipation
 3       C001    underparticipation,
               feature_group
 0         volatility_stress
 1                     other
 2            trend_momentum
 3             drawdown_path
 4             soxx_relative
 5          reversal_reentry
 6         volatility_regime
 7               price_level
 8  relative_strength_spread)

## Score Reward / Checkpoint Recipes

Full gate remains strict: all required rows must pass, and missing `wf_2020` blocks final retention. The selected row can only become a next-training recipe.

In [5]:
config = RewardCheckpointScreenConfig(
    target_relative_return=target_rel,
    target_max_dd=target_dd,
    relative_gap_weight=1.0,
    max_dd_gap_weight=10.0,
    max_dd_drift_weight=5.0,
    relative_drift_weight=0.25,
    missing_short_volshock_penalty=25.0,
    short_volshock_accuracy_floor=0.60,
    min_short_volshock_samples=10,
)

scored = score_reward_checkpoint_candidates(
    gate_gaps,
    slice_scores=slice_scores,
    poor_window_clusters=poor_window_clusters,
    feature_groups=feature_groups,
    reference_candidate=source_candidate,
    config=config,
)
summary_core = summarize_reward_checkpoint_screen(scored)
review_md = render_reward_checkpoint_screen_review(scored, summary_core)

scored.to_csv(CARD / "reward_checkpoint_scores.csv", index=False)
(CARD / "reward_checkpoint_review.md").write_text(review_md)
(CARD / "reward_checkpoint_summary.json").write_text(json.dumps(summary_core, indent=2, sort_keys=True) + "\n")
scored


,candidate,window_count,gate_pass_count,full_gate,reward_checkpoint_loss,gate_gap_loss,relative_drift_loss,max_dd_drift_loss,short_volshock_slice_loss,min_relative_return,max_max_dd,failure_mix,dominant_failure_type,required_objective_terms,short_volshock_target_samples,mean_short_volshock_accuracy
0,teacher_short_volshock_cap05_hit2,3,1,False,31.367859,31.256545,0.445257,0.0,0.0,0.0,15.107381,"coverage_missing:1,pass:1,joint_tradeoff:1",coverage_missing,"[""checkpoint_objective_loss"", ""drawdown_drift_...",300,0.953125
1,source_multi_obj_floor45_cap20_riskfirst,3,0,False,197.488279,197.488279,0.000000,0.0,0.0,0.0,30.626799,"drawdown_control:2,coverage_missing:1",drawdown_control,"[""checkpoint_objective_loss"", ""drawdown_drift_...",0,NaN


## Persist Selection And Summary

The summary keeps the distinction between recipe readiness and promotion. If the selected candidate lacks a full gate, `ppo_allowed_next` and `training_allowed_from_this_card` stay false.

In [6]:
selected = scored.iloc[0].to_dict() if not scored.empty else {}
selected_candidate_name = str(selected.get("candidate", "")) if selected else ""
selected_eval_rows = gate_gaps[
    gate_gaps["profile"].astype(str).eq(selected_candidate_name)
    & ~gate_gaps["failure_type"].astype(str).eq("coverage_missing")
]
required_terms = json.loads(str(selected.get("required_objective_terms", "[]"))) if selected else []
selection = {
    "experiment": "short_volshock_teacher_reward_checkpoint_screen_v1",
    "created_at": created_at,
    "role": "diagnose",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "short_volshock_metric_aligned_checkpoint_or_bc_teacher_v1",
    "source_teacher_screen": str(TEACHER.relative_to(ROOT)),
    "source_candidate": source_candidate,
    "selected_candidate": summary_core.get("selected_candidate", ""),
    "changed_layer": ["reward", "checkpoint_selection", "model_selection"],
    "target_windows": required_windows,
    "accepted_tradeoff": "none unless all required windows clear official gate and final retention is re-run",
    "hypothesis": "metric-aligned short-volshock teacher can define drawdown-drift checkpoint/reward terms without copying eval-window labels",
    "acceptance_criteria": {
        "official_eval_relative_return_pp": target_rel,
        "official_eval_max_dd_pct": target_dd,
        "final_required_windows": required_windows,
        "fold_local_targets_required_before_training": True,
    },
    "closest_historical_success_case": "short_volshock_cap05_hit2 passed wf_2022 official replay and was near-miss on wf_2023_2024",
    "closest_historical_failure_case": "short_volshock_reward_checkpoint_screen_v1 blocked plain short-volshock because wf_2022 drawdown and wf_2023_2024 joint tradeoff failed",
    "why_not_duplicate": "this screen uses the 2026-06-16 metric-aligned teacher handoff and explicit wf_2020 coverage blocker, not the older 2026-05 short_volshock_plain candidate",
    "decision": summary_core["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False if not bool(selected.get("full_gate", False)) else bool(summary_core.get("ppo_allowed_next")),
    "training_allowed_from_this_card": False,
    "checkpoint_recipe_allowed_next": not bool(selected.get("full_gate", False)),
    "leakage_guard": teacher_recipe["leakage_guard"],
    "do_not_use_as": teacher_recipe["do_not_use_as"],
    "required_objective_terms": required_terms,
    "next_mechanism": "implement_consumed_reward_or_checkpoint_recipe_then_fold_local_target_scope_audit",
}
summary = {
    "experiment": selection["experiment"],
    "created_at": created_at,
    "decision": selection["decision"],
    "selected_candidate": selection["selected_candidate"],
    "promotion_allowed": selection["promotion_allowed"],
    "actor_retention_allowed": selection["actor_retention_allowed"],
    "ppo_allowed_next": selection["ppo_allowed_next"],
    "training_allowed_from_this_card": selection["training_allowed_from_this_card"],
    "checkpoint_recipe_allowed_next": selection["checkpoint_recipe_allowed_next"],
    "target_relative_return_pp": target_rel,
    "target_max_dd_pct": target_dd,
    "final_required_windows": required_windows,
    "missing_final_required_windows": sorted(missing_windows),
    "selected_reward_checkpoint_loss": float(selected.get("reward_checkpoint_loss", 0.0) or 0.0),
    "selected_gate_gap_loss": float(selected.get("gate_gap_loss", 0.0) or 0.0),
    "selected_min_relative_return_pp": float(selected.get("min_relative_return", 0.0) or 0.0),
    "selected_max_max_dd_pct": float(selected.get("max_max_dd", 0.0) or 0.0),
    "selected_eval_window_min_relative_return_pp": float(selected_eval_rows["relative_return"].min()) if not selected_eval_rows.empty else None,
    "selected_eval_window_max_dd_pct": float(selected_eval_rows["max_dd"].max()) if not selected_eval_rows.empty else None,
    "selected_failure_mix": str(selected.get("failure_mix", "")),
    "required_objective_terms": required_terms,
    "source_teacher_decision": teacher_summary["decision"],
    "source_teacher_selected_candidate": teacher_summary["selected_candidate"],
    "source_teacher_max_relative_gap_pp": teacher_summary["selected_max_relative_gap_pp"],
    "source_teacher_max_dd_gap_pct": teacher_summary["selected_max_dd_gap_pct"],
    "trace_interpretation": "low_order_absorption_use_state_checkpoint_not_more_overlay",
    "next_mechanism": selection["next_mechanism"],
    "sidecars": [
        "gate_gaps.csv",
        "slice_scores.csv",
        "feature_groups.csv",
        "reward_checkpoint_scores.csv",
        "reward_checkpoint_summary.json",
        "selection.json",
        "summary.json",
        "reward_checkpoint_review.md",
        "conclusion.md",
    ],
}
conclusion_lines = [
    f"# {selection['experiment']} Conclusion",
    "",
    f"Decision: `{summary['decision']}`.",
    "",
    f"Selected candidate `{summary['selected_candidate']}` has evaluated-window min relative return {summary['selected_eval_window_min_relative_return_pp']:.2f}pp and max MaxDD {summary['selected_eval_window_max_dd_pct']:.2f}%.",
    f"Coverage-scored min relative return is {summary['selected_min_relative_return_pp']:.2f}pp because missing windows are explicit failure rows.",
    f"Missing final windows: {', '.join(summary['missing_final_required_windows']) or 'none'}.",
    f"Required terms: {', '.join(required_terms)}.",
    "",
    "No promotion, no actor retention, and no longer PPO are allowed from this card. The teacher is usable only as checkpoint-scoring or fold-local target-generation reference after a target-scope leakage audit.",
    "",
    f"Next action: `{summary['next_mechanism']}`.",
]
conclusion_md = "\n".join(conclusion_lines) + "\n"

(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
(CARD / "conclusion.md").write_text(conclusion_md)

summary


{'experiment': 'short_volshock_teacher_reward_checkpoint_screen_v1',
 'created_at': '2026-06-16T03:59:21+00:00',
 'decision': 'reward_checkpoint_screen_no_promotion',
 'selected_candidate': 'teacher_short_volshock_cap05_hit2',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'ppo_allowed_next': False,
 'training_allowed_from_this_card': False,
 'checkpoint_recipe_allowed_next': True,
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'final_required_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'missing_final_required_windows': ['wf_2020'],
 'selected_reward_checkpoint_loss': 31.367858850544028,
 'selected_gate_gap_loss': 31.25654454559548,
 'selected_min_relative_return_pp': 0.0,
 'selected_max_max_dd_pct': 15.1073807852534,
 'selected_eval_window_min_relative_return_pp': 29.81726330693851,
 'selected_eval_window_max_dd_pct': 15.1073807852534,
 'selected_failure_mix': 'coverage_missing:1,pass:1,joint_tradeoff:1',
 'required_objective_terms': ['checkp

In [7]:
from IPython.display import Markdown

Markdown(conclusion_md)


# short_volshock_teacher_reward_checkpoint_screen_v1 Conclusion

Decision: `reward_checkpoint_screen_no_promotion`.

Selected candidate `teacher_short_volshock_cap05_hit2` has evaluated-window min relative return 29.82pp and max MaxDD 15.11%.
Coverage-scored min relative return is 0.00pp because missing windows are explicit failure rows.
Missing final windows: wf_2020.
Required terms: checkpoint_objective_loss, drawdown_drift_penalty, short_volshock_state_deleverage_consistency, trend_preservation_floor.

No promotion, no actor retention, and no longer PPO are allowed from this card. The teacher is usable only as checkpoint-scoring or fold-local target-generation reference after a target-scope leakage audit.

Next action: `implement_consumed_reward_or_checkpoint_recipe_then_fold_local_target_scope_audit`.
